In [1]:
import pandas as pd 
import numpy as np


In [2]:
df_historico = pd.read_csv("../data/nfl_partidos_historico.csv")
df_2026 = pd.read_csv("../data/nfl_calendario_2026.csv")

print(f"Histórico: {df_historico.shape[0]} partidos.")
print(f"Calendario 2026: {df_2026.shape[0]} partidos.")

Histórico: 5698 partidos.
Calendario 2026: 272 partidos.


In [3]:
mapeo_equipos = {
    'STL': 'LA',
    'SD': 'LAC',
    'OAK': 'LV',
    'WSH': 'WAS',
    'LAR': 'LA'
}

def estandarizar(df, mapeo):
    df_clean = df.copy()
    df_clean['home_team'] = df_clean['home_team'].map(mapeo).fillna(df_clean['home_team'])
    df_clean['away_team'] = df_clean['away_team'].map(mapeo).fillna(df_clean['away_team'])
    return df_clean

df_historico_clean = estandarizar(df_historico, mapeo_equipos)
df_2026_clean = estandarizar(df_2026, mapeo_equipos)

equipos_hist_clean = set(df_historico_clean['home_team'].unique()) | set(df_historico_clean['away_team'].unique())
equipos_2026_clean = set(df_2026_clean['home_team'].unique()) | set(df_2026_clean['away_team'].unique())

print("\n--- REVISIÓN DE EQUIPOS ---")
if equipos_hist_clean == equipos_2026_clean:
    print(f"✅ ¡Éxito! Ambos datasets ahora manejan exactamente las mismas {len(equipos_2026_clean)} franquicias.")
else:
    print("⚠️ Aún hay diferencias:")
    print(f"En histórico pero no en 2026: {equipos_hist_clean - equipos_2026_clean}")
    print(f"En 2026 pero no en histórico: {equipos_2026_clean - equipos_hist_clean}")




--- REVISIÓN DE EQUIPOS ---
✅ ¡Éxito! Ambos datasets ahora manejan exactamente las mismas 32 franquicias.


In [5]:
columnas_elo = [
    'season', 
    'week', 
    'game_type', 
    'home_team', 
    'away_team', 
    'home_score', 
    'away_score', 
    'result' # home_score - away_score
]

df_historico_elo = df_historico_clean[columnas_elo].copy()


df_historico_elo = df_historico_elo.sort_values(by=['season', 'week'])

partidos_antes = len(df_historico_elo)
df_historico_elo = df_historico_elo.dropna(subset=['home_score', 'away_score'])
partidos_despues = len(df_historico_elo)

print(f"🧹 Partidos eliminados por valores nulos (cancelaciones): {partidos_antes - partidos_despues}")

print("\n--- DATASET LISTO PARA EL MODELO ---")
display(df_historico_elo.head())

🧹 Partidos eliminados por valores nulos (cancelaciones): 0

--- DATASET LISTO PARA EL MODELO ---


,season,week,game_type,home_team,away_team,home_score,away_score,result
0,2005,1,REG,NE,LV,30.0,20.0,10.0
1,2005,1,REG,BUF,HOU,22.0,7.0,15.0
2,2005,1,REG,CAR,NO,20.0,23.0,-3.0
3,2005,1,REG,CLE,CIN,13.0,27.0,-14.0
4,2005,1,REG,JAX,SEA,26.0,14.0,12.0


In [6]:
ruta_hist_clean = "../data/nfl_historico_clean.csv"
ruta_2026_clean = "../data/nfl_calendario_2026_clean.csv"

df_historico_elo.to_csv(ruta_hist_clean, index=False)
# Para el 2026, mantenemos las mismas columnas relevantes
df_2026_elo = df_2026_clean[columnas_elo].copy()
df_2026_elo.to_csv(ruta_2026_clean, index=False)

print("💾 Archivos limpios guardados exitosamente. Listos para el Día 3.")

💾 Archivos limpios guardados exitosamente. Listos para el Día 3.
